# glcuda prefill profiling — NO llama.cpp build

The llama.cpp CUDA build takes ~10-15 min every session. We don't need it:
decode is already at/above llama.cpp parity (measured 1.0-1.27x on the T4),
and the open question is glcuda PREFILL — specifically the FFN breakdown.

This notebook builds ONLY glcuda (~2-3 min) and runs the per-op prefill
profiler. llama.cpp prefill baseline (from earlier same-T4 runs, for
reference only): Q8_0 ~1400, Q4_K_M ~1200, Q4_0 ~1320 tok/s.

Runtime: GPU T4 + Internet on. ~6 min incl. one 4.4 GB model download.


## 0 · GPU + Rust toolchain

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import os, subprocess
# Kaggle path adapter (harmless on Colab).
if os.path.exists('/kaggle') and not os.path.exists('/content'):
    subprocess.run(['ln','-sfn','/kaggle/working','/content'], check=True)
    print('linked /content -> /kaggle/working')
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!cargo --version


## 1 · Clone glcuda + build the runner (glcuda only, ~2-3 min)

In [ ]:
%cd /content
if not os.path.exists('/content/gwenland-ai'):
    !git clone -q https://github.com/JinXSuperSolo/gwenland-ai
%cd /content/gwenland-ai
!git pull --ff-only 2>&1 | tail -1

# run_glcuda driver: prints prefill/decode tok/s; sampling pinned.
example = r"""
use std::time::Instant;
use glcore::engine_trait::{GlEngine, InferInput};
use glcuda::GlcudaEngine;
fn main() -> Result<(), Box<dyn std::error::Error>> {
    let path = std::env::args().nth(1).expect("usage: run_glcuda <model.gguf> [prompt]");
    let prompt = std::env::args().nth(2)
        .unwrap_or_else(|| "Explain what a GPU is in one sentence.".to_string());
    let mut engine = GlcudaEngine::new();
    engine.init()?;
    let t = Instant::now();
    engine.load_model(&path)?;
    println!("[load] {:.2}s", t.elapsed().as_secs_f64());
    let ids = engine.encode_chat(&prompt)?;
    println!("[prompt] {} tokens", ids.len());
    let input = InferInput { token_ids: ids, max_new_tokens: 128, temperature: 0.7,
        top_k: 40, top_p: 0.95, repeat_penalty: 1.1 };
    let out = engine.stream(input, &|_id,_p| {})?;
    println!("GLCUDA prefill {:.1} tok/s | decode {:.1} tok/s",
        out.prompt_tokens as f64 / (out.prefill_ms/1e3).max(1e-9),
        out.tokens_generated as f64 / (out.generation_ms/1e3).max(1e-9));
    Ok(())
}
"""
os.makedirs('/content/gwenland-ai/glcuda/examples', exist_ok=True)
open('/content/gwenland-ai/glcuda/examples/run_glcuda.rs','w').write(example)
mani = '/content/gwenland-ai/glcuda/Cargo.toml'
txt = open(mani).read()
if '[[example]]' not in txt:
    open(mani,'a').write('\n[[example]]\nname = "run_glcuda"\npath = "examples/run_glcuda.rs"\n')
!cargo build --release -p glcuda --example run_glcuda 2>&1 | tail -3


## 2 · Model — Qwen2.5-7B Q8_0 (the FFN-profiling target)

In [ ]:
%cd /content/gwenland-ai
if not os.path.exists('q8.gguf'):
    os.system('wget -q -O q8.gguf https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q8_0.gguf')
!ls -lh q8.gguf


## 3 · ⭐ The FFN breakdown — this is the number we need

`GLCUDA_PROFILE_PREFILL=1` prints two lines: the coarse qkv/attn/ffn split,
and the FFN detail (gate+up GEMM vs down+o GEMM vs elementwise). The biggest
FFN sub-bucket is the next optimization target — measure, don't guess.


In [ ]:
%cd /content/gwenland-ai
import os, subprocess
os.environ.pop('GLCUDA_NO_MMA', None)
BASE = 'Explain how quantum computing differs from classical computing, covering qubits, superposition, entanglement, and key algorithms. '
PROMPT = (BASE * 24).strip()   # ~500 tokens, like llama-bench pp512
env = {**os.environ, 'GLCUDA_PROFILE_PREFILL': '1'}
r = subprocess.run(['target/release/examples/run_glcuda','q8.gguf',PROMPT],
                   capture_output=True, text=True, env=env)
for line in r.stderr.splitlines():
    if 'prefill split' in line or 'ffn detail' in line:
        print(line)
for line in r.stdout.splitlines():
    if 'GLCUDA' in line or '[load]' in line or '[prompt]' in line:
        print(line)


## 4 · (Optional) glcuda-only prefill+decode table, all 3 formats

No llama.cpp — just glcuda's own numbers vs the fixed baseline in the header.
Downloads Q4_K_M and a pure Q4_0 (converted with glcuda's loader path needs
a real Q4_0 file; skip if you only care about the FFN breakdown above).


In [ ]:
%cd /content/gwenland-ai
import re
def glcuda(model):
    r = subprocess.run(['target/release/examples/run_glcuda', model, PROMPT],
                       capture_output=True, text=True)
    m = re.search(r'GLCUDA prefill ([\d.]+) tok/s \| decode ([\d.]+)', r.stdout)
    if not m:
        print('  parse miss', model, '::', r.stderr.strip()[-200:])
    return (float(m.group(1)), float(m.group(2))) if m else (0.0, 0.0)

# llama.cpp baseline from earlier same-T4 runs (reference only, not re-run).
LLAMA = {'Q8_0': (1400, 29), 'Q4_K_M': (1200, 26), 'Q4_0': (1320, 38)}
print(f'{"format":<8}{"glcuda pf":>10}{"llama pf":>10}{"pf gap":>8}{"glcuda dec":>12}{"llama dec":>11}')
print('-'*60)
for fmt, model in [('Q8_0','q8.gguf')]:   # add q4km.gguf / q4_0 files if downloaded
    if not os.path.exists(model):
        print(f'{fmt:<8} (missing)'); continue
    gp, gd = glcuda(model)
    lp, ld = LLAMA[fmt]
    print(f'{fmt:<8}{gp:>10.1f}{lp:>10.0f}{lp/max(gp,1e-9):>7.1f}x{gd:>12.1f}{ld:>11.0f}')
